In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the cleaned SEDS dataset
#df = pd.read_csv('df_seds_cleaned.csv')

#To load the dataset in Colab, comment out the line above and the categories = line below and uncomment this section:
from google.colab import drive
drive.mount('/content/gdrive')

path = 'gdrive/MyDrive/SIADS_696_Milestone_II/Datasets/EIA_files'
df = pd.read_csv(f'{path}/df_seds_cleaned.csv')

categories = pd.read_excel(f'{path}/SEDS_categories.xlsx')

# Load the SEDS category/description file
#categories = pd.read_excel('SEDS_categories.xlsx')

df.head()

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


FileNotFoundError: [Errno 2] No such file or directory: '/content/gdrive/MyDrive/SIADS_696_Milestone_II/Datasets/EIA_files/df_seds_cleaned.csv'

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print("Number of states:", df['State'].nunique())
print("Number of years:", df['Year'].nunique())
print("Year range:", df['Year'].min(), "-", df['Year'].max())

In [ ]:
# Examine data types and non-null counts
df.info()

In [ ]:
print("Total missing values:", df.isnull().sum().sum())
print("Duplicate State-Year observations:", df.duplicated(['State', 'Year']).sum())

In [ ]:
# Identify the energy variables
energy_cols = df.columns[2:]

print("Number of energy variables:", len(energy_cols))
print(energy_cols.tolist())

In [ ]:
# View the category/description information
categories.columns

In [ ]:
# Create a smaller lookup table for the MSN variables
variable_info = categories[['MSN Code', 'Description', 'Unit']].copy()

variable_info.head()

In [ ]:
summary_stats = df[energy_cols].describe().T

summary_stats

In [ ]:
# Calculate the percentage of observations equal to zero
zero_percentage = (
    df[energy_cols]
    .eq(0)
    .mean() * 100
).sort_values(ascending=False)

zero_percentage.head(10)

In [ ]:
# Add descriptions to the zero-value summary
zero_summary = (
    zero_percentage
    .rename('Zero_Percentage')
    .reset_index()
    .rename(columns={'index': 'MSN Code'})
)

zero_summary = zero_summary.merge(
    variable_info,
    on='MSN Code',
    how='left'
)

zero_summary.head(10)

In [ ]:
# Count negative observations for each energy variable
negative_counts = (
    (df[energy_cols] < 0)
    .sum()
    .sort_values(ascending=False)
)

negative_counts[negative_counts > 0]

In [ ]:
# Describe the negative-value summary
negative_summary = (
    negative_counts[negative_counts > 0]
    .rename('Negative_Count')
    .reset_index()
    .rename(columns={'index': 'MSN Code'})
)

negative_summary = negative_summary.merge(
    variable_info,
    on='MSN Code',
    how='left'
)

negative_summary

In [ ]:
# Representative production variables
production_vars = [
    'PAPRB',
    'NGMPB',
    'CLPRB',
    'NUPRB',
    'HYPRB',
    'WYPRB',
    'SOPRB'
]

# Check that the selected variables exist in the dataset
[var for var in production_vars if var not in df.columns]

In [ ]:
# Histograms for selected production variables
for var in production_vars:
    plt.figure(figsize=(8, 5))
    plt.hist(df[var], bins=30)
    plt.title(f'Distribution of {var}')
    plt.xlabel('Billion Btu')
    plt.ylabel('Number of Observations')
    plt.show()

In [ ]:
# Aggregate selected production variables across geographic entities by year
yearly_energy = df.groupby('Year')[production_vars].sum()

yearly_energy.head()

In [ ]:
# Plot each selected production variable over time
for var in production_vars:
    plt.figure(figsize=(10, 5))
    plt.plot(yearly_energy.index, yearly_energy[var])
    plt.title(f'{var} Production Over Time')
    plt.xlabel('Year')
    plt.ylabel('Billion Btu')
    plt.show()

In [ ]:
# Average crude oil production by state
state_oil = (
    df.groupby('State')['PAPRB']
    .mean()
    .sort_values(ascending=False)
)

state_oil.head(10)

In [ ]:
# Plot the 10 states with the highest average crude oil production
state_oil.head(10).plot(
    kind='bar',
    figsize=(10, 5)
)

plt.title('Top 10 States by Average Crude Oil Production')
plt.xlabel('State')
plt.ylabel('Average Billion Btu')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Compare production and transportation consumption
plt.figure(figsize=(8, 6))

plt.scatter(
    df['PAPRB'],
    df['PAACB'],
    alpha=0.5
)

plt.title('Crude Oil Production vs. Petroleum Transportation Consumption')
plt.xlabel('Crude Oil Production (Billion Btu)')
plt.ylabel('Petroleum Transportation Consumption (Billion Btu)')
plt.show()

In [ ]:
correlation_matrix = df[energy_cols].corr()

correlation_matrix.head()

In [ ]:
# Visualize the correlation matrix
plt.figure(figsize=(14, 12))

plt.imshow(correlation_matrix, aspect='auto')
plt.colorbar()

plt.xticks(
    range(len(energy_cols)),
    energy_cols,
    rotation=90
)

plt.yticks(
    range(len(energy_cols)),
    energy_cols
)

plt.title('Correlation Between SEDS Energy Variables')
plt.tight_layout()
plt.show()

In [ ]:
# Look for the strongest positive and negative pairwise correlations and show them
corr_pairs = (
    correlation_matrix
    .where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
)

print("Strongest positive correlations:")
display(corr_pairs.head(10))

print("\nStrongest negative correlations:")
display(corr_pairs.tail(10))

In [ ]:
# Ten largest crude oil production observations
df.nlargest(
    10,
    'PAPRB'
)[['State', 'Year', 'PAPRB']]

In [ ]:
# Ten largest natural gas production observations
df.nlargest(
    10,
    'NGMPB'
)[['State', 'Year', 'NGMPB']]

In [ ]:
# Average production profile for each state
state_energy = df.groupby('State')[production_vars].mean()

state_energy.head()